In [2]:
import sys
from pathlib import Path

from matplotlib.animation import FuncAnimation
from pathlib import Path
python_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(python_dir))

import integration.rhs as rhs
import numpy as np
import matplotlib.pyplot as plt
import h5py
import scipy.special as sp
import FourierSeries as fs
from scipy.integrate import solve_ivp
import os
from matplotlib.widgets import Slider
from scipy.linalg import null_space
from scipy.interpolate import interp1d


In [30]:
path = r"\\data03.physics.leidenuniv.nl\pi-bouwmeester\optomechanics\Hidde\phd\nonlinear_dynamics_thin_film_helium\writing\mode_locking_paper\code\sim_verification\mechanical_lasing\experiment\lasing_one_mode\380mVPM"

data = {}

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".txt"):
            file_path = os.path.join(root, file)
            time, transmission = np.loadtxt(file_path, delimiter=' ', unpack=True)
            data[file_path] = {'time': time, 'transmission': transmission}

In [31]:
def exponential_moving_average(data, alpha):
    """
    Apply exponential moving average to a numpy array.
    
    Parameters:
    -----------
    data : np.ndarray
        Input array to smooth
    alpha : float
        Smoothing factor between 0 and 1. 
        Higher alpha = less smoothing, lower alpha = more smoothing.
    
    Returns:
    --------
    np.ndarray
        Smoothed array
    """
    smoothed = np.zeros_like(data)
    smoothed[0] = data[0]
    for i in range(1, len(data)):
        smoothed[i] = alpha * data[i] + (1 - alpha) * smoothed[i - 1]
    return smoothed

In [8]:
%matplotlib Qt

In [44]:
from plots import plot_phase_diagram

fig, ax = plt.subplots(2,1, figsize=(12, 6))
ax_pd, ax_ts = ax
max_plots = 3

for i, (file_path, data_dict) in enumerate(data.items()):
    time = data_dict['time']
    transmission = data_dict['transmission']
    ## smooth the data using an exponential moving average
    smoothed = exponential_moving_average(transmission, alpha=0.2)
    plot_phase_diagram(smoothed, time, 1e3* 1.0e-6, ax_pd, ax_ts, label=file_path.split("\\")[-2])
    if i >= max_plots:
        break
    # ax_ts.plot(time, transmission, label=file_path.split("\\")[-1])
# ax_ts.set_xlabel("Time (ms)")
ax_pd.set_title("Phase Diagram")
ax_ts.set_title("Time Series")
ax_pd.legend()
ax_ts.legend()

ax_pd.set_xlabel("Transmission")
ax_pd.set_ylabel("d(Transmission)/dt")

ax_ts.set_xlabel("Time (ms)")
ax_ts.set_ylabel("Transmission")


Text(0, 0.5, 'Transmission')

In [ ]:
data.items()

dict_items([('\\\\data03.physics.leidenuniv.nl\\pi-bouwmeester\\optomechanics\\Hidde\\phd\\nonlinear_dynamics_thin_film_helium\\writing\\mode_locking_paper\\code\\sim_verification\\mechanical_lasing\\experiment\\lasing_one_mode\\850mVPM\\80mVdet\\timetrace.txt', {'time': array([-4.34242286e-02, -4.34198857e-02, -4.34155429e-02, ...,
       -8.68571429e-06, -4.34285714e-06,  0.00000000e+00]), 'transmission': array([0.04753607, 0.05126472, 0.05651009, ..., 0.25324663, 0.26014393,
       0.26438064])}), ('\\\\data03.physics.leidenuniv.nl\\pi-bouwmeester\\optomechanics\\Hidde\\phd\\nonlinear_dynamics_thin_film_helium\\writing\\mode_locking_paper\\code\\sim_verification\\mechanical_lasing\\experiment\\lasing_one_mode\\850mVPM\\60mVdet\\timetrace.txt', {'time': array([-4.34242286e-02, -4.34198857e-02, -4.34155429e-02, ...,
       -8.68571429e-06, -4.34285714e-06,  0.00000000e+00]), 'transmission': array([0.04683372, 0.04760387, 0.05003019, ..., 0.09192674, 0.091931  ,
       0.08697702])}), 